# Variant C — Multi-Task Learning (Classify + Explain)

**Architecture:** DeBERTa-v3-base with dual heads:
1. Classification head: NLI label prediction from [CLS]
2. MLM head: masked token prediction on explanation tokens only

**Joint Loss:** `0.7 × CE_label + 0.3 × MLM_explanation`

At inference, the MLM head is discarded.

In [ ]:
# Cell 1: Install dependencies
!pip install -q transformers datasets accelerate torch

In [ ]:
# Cell 2: Mount Google Drive and set project path
from google.colab import drive
drive.mount("/content/drive")

import os
import sys

# Set this to where your project folder lives on Google Drive
PROJECT_ROOT = "/content/drive/MyDrive/Old-Explanations-New-Rationales-Bridging-e-SNLI-and-LLM-Chain-of-Thought-in-Encoder-Based-NLI"

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

print("Working directory:", os.getcwd())
print("Contents:", os.listdir("."))

In [ ]:
# Cell 3: Verify GPU
import torch

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:     ", torch.cuda.get_device_name(0))
    print("GPU memory:   ", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    print("WARNING: No GPU detected.")
    print("Go to Runtime > Change runtime type > GPU > Save and re-run.")

In [ ]:
# Cell 4: Load config and data (with subsampling for Colab time budget)
from training.config import VariantCConfig
from data.preprocess import load_esnli_train, load_esnli_split

config = VariantCConfig()

print("Loading training data...")
train_df_full = load_esnli_train(config)

# Subsample to fit within a 3-hour Colab session.
# 100K examples = ~9,375 optimizer steps = ~1.5-2.5 hrs on T4.
# If speed is below 0.5 it/s after a few minutes, reduce to 50_000.
MAX_TRAIN_SAMPLES = 100_000
train_df = train_df_full.sample(n=MAX_TRAIN_SAMPLES, random_state=42).reset_index(drop=True)

print("Loading dev data...")
dev_df = load_esnli_split(config, "dev")

print(f"\nFull train set:  {len(train_df_full):,}")
print(f"Using:           {len(train_df):,} ({MAX_TRAIN_SAMPLES / len(train_df_full) * 100:.0f}%)")
print(f"Dev set:         {len(dev_df):,}")
print(f"\nSample row:")
print(f"  Premise:     {train_df['Sentence1'].iloc[0]}")
print(f"  Hypothesis:  {train_df['Sentence2'].iloc[0]}")
print(f"  Explanation: {train_df['Explanation_1'].iloc[0]}")
print(f"  Label:       {train_df['gold_label'].iloc[0]}")

In [ ]:
# Cell 5: Build tokenizer, datasets, and model
from models.common import load_tokenizer
from models.variant_c import DeBERTaForMultiTask
from data.preprocess import ESNLIMultiTaskDataset

print("Loading tokenizer...")
tokenizer = load_tokenizer(config.model_name)
print(f"  is_fast: {tokenizer.is_fast}")

print("\nBuilding datasets...")
train_dataset = ESNLIMultiTaskDataset(train_df, tokenizer, config, is_train=True)
eval_dataset  = ESNLIMultiTaskDataset(dev_df,   tokenizer, config, is_train=False)
print(f"  Train dataset: {len(train_dataset):,} examples")
print(f"  Eval dataset:  {len(eval_dataset):,} examples")

# Sanity check: verify masking is working
sample = train_dataset[0]
masked = (sample['mlm_labels'] != -100).sum().item()
print(f"\nSanity check on one sample:")
print(f"  input_ids shape:   {sample['input_ids'].shape}")
print(f"  mlm_labels shape:  {sample['mlm_labels'].shape}")
print(f"  label:             {sample['labels'].item()} ({config.id2label[sample['labels'].item()]})")
print(f"  tokens masked:     {masked}  (must be > 0)")

print(f"\nLoading model: {config.model_name}")
print("  (Downloads ~700MB on first run, ~30 seconds on repeat runs)")
model = DeBERTaForMultiTask(
    model_name=config.model_name,
    num_labels=config.num_labels,
    alpha=config.alpha,
)
total = sum(p.numel() for p in model.parameters())
print(f"  Total parameters: {total:,}")
print(f"  Alpha (cls weight): {config.alpha}  |  MLM weight: {1 - config.alpha}")

In [ ]:
# Cell 6: Check for existing checkpoint (session recovery)
import glob
import shutil

checkpoint_dir = str(config.get_path("checkpoint_dir"))
os.makedirs(checkpoint_dir, exist_ok=True)

resume_from = None
existing = sorted(glob.glob(os.path.join(checkpoint_dir, "checkpoint-*")))

for ckpt in existing:
    model_pt = os.path.join(ckpt, "model.pt")
    if os.path.isfile(model_pt):
        resume_from = ckpt   # valid checkpoint — has our model.pt
    else:
        print(f"Removing invalid checkpoint (no model.pt): {ckpt}")
        shutil.rmtree(ckpt)  # delete it so the Trainer doesn't try to load it

if resume_from:
    print(f"Found valid checkpoint: {resume_from}")
    print("Training will resume from this point.")
else:
    print("No valid checkpoint found — training from scratch.")

print(f"\nCheckpoint directory: {checkpoint_dir}")

In [ ]:
# Cell 7: Train
import numpy as np
from transformers import TrainingArguments
from training.train import MultiTaskTrainer, compute_metrics

# Fix 1: prevent GPU memory fragmentation
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# Fix 2: release any leftover GPU memory from earlier cells
torch.cuda.empty_cache()

# Fix 3: ensure all parameters are float32 so the fp16 gradient scaler works correctly
model = model.float()

steps_per_epoch = len(train_dataset) // (config.per_device_train_batch_size * config.gradient_accumulation_steps)
total_steps     = steps_per_epoch * config.num_train_epochs

print("Training config:")
print(f"  Batch size (per device):  {config.per_device_train_batch_size}")
print(f"  Gradient accumulation:    {config.gradient_accumulation_steps}")
print(f"  Effective batch size:     {config.per_device_train_batch_size * config.gradient_accumulation_steps}")
print(f"  Steps per epoch:          {steps_per_epoch}")
print(f"  Total optimizer steps:    {total_steps}")
print(f"  Warmup steps:             {config.warmup_steps}")
print()
print("Watch the it/s value in the progress bar:")
print("  > 0.8 it/s  → on track, let it run")
print("  0.5-0.8     → will take 3-5 hrs, still okay")
print("  < 0.5 it/s  → stop, go back to Cell 4, set MAX_TRAIN_SAMPLES = 50_000")
print()

training_args = TrainingArguments(
    output_dir=checkpoint_dir,
    num_train_epochs=config.num_train_epochs,
    per_device_train_batch_size=config.per_device_train_batch_size,
    per_device_eval_batch_size=config.per_device_eval_batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    learning_rate=config.learning_rate,
    weight_decay=config.weight_decay,
    warmup_steps=config.warmup_steps,
    fp16=config.fp16,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=config.save_total_limit,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    dataloader_num_workers=0,
    logging_steps=50,
    report_to="none",
    remove_unused_columns=False,
)

trainer = MultiTaskTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)
trainer._variant_c_config = config

trainer.train(resume_from_checkpoint=resume_from)
print("\nTraining complete!")

In [ ]:
# Cell 8: Save final model and config to Drive
import json

final_dir = str(config.get_path("output_dir"))
os.makedirs(final_dir, exist_ok=True)

model_path = os.path.join(final_dir, "model.pt")
torch.save(model.state_dict(), model_path)
print(f"Model saved:  {model_path}")

config_path = os.path.join(final_dir, "variant_c_config.json")
with open(config_path, "w") as f:
    json.dump(vars(config), f, indent=2, default=str)
print(f"Config saved: {config_path}")

In [ ]:
# Cell 9: Evaluate on e-SNLI test (premise + hypothesis only — no explanation at inference)
from evaluation.evaluate import evaluate_dataset
from data.preprocess import NLIDataset, load_esnli_split

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

print("Loading e-SNLI test set...")
test_df = load_esnli_split(config, "test")

test_dataset = NLIDataset(
    premises=test_df["Sentence1"].tolist(),
    hypotheses=test_df["Sentence2"].tolist(),
    labels=[config.label2id[lbl] for lbl in test_df["gold_label"].tolist()],
    tokenizer=tokenizer,
    max_length=config.max_seq_length,
)

print(f"Evaluating on {len(test_dataset):,} examples...")
acc = evaluate_dataset(model, test_dataset, batch_size=config.per_device_eval_batch_size, device=device)
print(f"\ne-SNLI test accuracy: {acc:.4f}  ({acc * 100:.2f}%)")

In [ ]:
# Cell 10: Cross-domain evaluation (SNLI, MultiNLI, ANLI)
from data.preprocess import load_snli_test, load_multinli, load_anli
import json

results = {"esnli_test": acc}

print("Evaluating SNLI test...")
snli = load_snli_test(tokenizer, config.max_seq_length)
results["snli_test"] = evaluate_dataset(model, snli, config.per_device_eval_batch_size, device)
print(f"  {results['snli_test']:.4f}")

print("Evaluating MultiNLI matched...")
mnli_m = load_multinli(tokenizer, split="validation_matched", max_length=config.max_seq_length)
results["multinli_matched"] = evaluate_dataset(model, mnli_m, config.per_device_eval_batch_size, device)
print(f"  {results['multinli_matched']:.4f}")

print("Evaluating MultiNLI mismatched...")
mnli_mm = load_multinli(tokenizer, split="validation_mismatched", max_length=config.max_seq_length)
results["multinli_mismatched"] = evaluate_dataset(model, mnli_mm, config.per_device_eval_batch_size, device)
print(f"  {results['multinli_mismatched']:.4f}")

for r in ("r1", "r2", "r3"):
    print(f"Evaluating ANLI {r}...")
    anli = load_anli(tokenizer, round_tag=r, max_length=config.max_seq_length)
    results[f"anli_{r}"] = evaluate_dataset(model, anli, config.per_device_eval_batch_size, device)
    print(f"  {results[f'anli_{r}']:.4f}")

print("\n" + "=" * 45)
print(f"{'Benchmark':<25} {'Accuracy':>10}")
print("=" * 45)
for k, v in results.items():
    print(f"{k:<25} {v * 100:>9.2f}%")

# Save results to Drive
results_path = os.path.join(final_dir, "eval_results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"\nResults saved to: {results_path}")